# Spoken digits against a tuned drive

In [1]:
import os, sys, subprocess, numpy as np
CODE = os.path.abspath(os.path.join(os.getcwd(), '..', 'code')); sys.path.insert(0, CODE); os.chdir(CODE); os.environ['OMP_NUM_THREADS'] = '1'
import matplotlib; matplotlib.use('Agg')
from IPython.display import Image, display
RUN = False   # set True to recompute the data of this notebook (cost given in the Compute section)
def show(pdf, dpi=110):
    base = '/tmp/' + os.path.basename(pdf)[:-4]; subprocess.run(['pdftoppm', '-r', str(dpi), '-png', '-singlefile', pdf, base], check=True); display(Image(filename=base + '.png'))
ld = lambda n: np.load(os.path.join('data', n), allow_pickle=True)


## Problem
Does squeezing improve spoken-digit recognition once the drive amplitude and frequency are tuned?
## Method (`rev_asr2.py`, protocol of `run_asr.py`, features from `data/asr_features.npz`)
Operating points selected by the first-fold training loss only: drive grid eps in {0.1, 0.2, 0.4} x delta in {-0.5..0.5}; squeezing scan from the tuned base, (r, r_e) in {0, 0.25, 0.5}^2, dth in {0, pi}, phi_d in {0, pi/2}; photon budget 2.5; five-fold WER at the selected points. Compute ~30 min.

In [2]:
if RUN: subprocess.run([sys.executable, 'rev_asr2.py'], check=True)

In [3]:
a = ld('rev_asr2.npz'); W = a['wer5']
for lab, w in zip(['fabricated', 'tuned drive', 'tuned + squeezed'], W): print(lab.ljust(18), 'WER per fold', (100 * w[:, 1]).round(0), 'mean %.1f%%' % (100 * w[:, 1].mean()))
print('selected (eps, delta, phi_d, r, r_e, dth):', a['sel'].round(3))
S = a['scan']; i = S[:, 4].argmin(); print('lowest scanned loss', S[i, 4], 'at', S[i, :4].round(3), 'nmax %.2f (budget 2.5)' % S[i, 7])

fabricated         WER per fold [ 7.  9. 16. 14. 14.] mean 12.0%
tuned drive        WER per fold [ 7.  6. 10. 13. 10.] mean 9.2%
tuned + squeezed   WER per fold [ 9.  4. 10. 12.  8.] mean 8.6%
selected (eps, delta, phi_d, r, r_e, dth): [ 0.4   -0.25   0.     0.25   0.5    3.142]
lowest scanned loss 0.006952236633666937 at [1.571 0.25  0.25  3.142] nmax 3.19 (budget 2.5)


The WER gain is a drive-tuning gain; squeezing the tuned device changes the WER by less than the fold resolution.

## Control: drive amplitude only (`run_asr_base.py`)
Holding the drive frequency at its fabricated value and scanning only the amplitude isolates which part of the drive matters. Amplitude alone is inert; squeezing from the amplitude-only base helps only through intracavity squeezing, whose cavity frequency $\Omega_a=\omega_a/\cosh 2r$ is the detuning the tuned drive supplies directly.

In [4]:
b = ld('asr_base.npz')
print('amplitude scan at zero squeezing (eps, fold-1 loss, fold-1 WER):', [(round(float(r[0]), 2), round(float(r[1]), 5), round(float(r[2]), 2)) for r in b['eps']])
print('5-fold WER: amplitude-optimized base (eps=%.2f) %.1f%% -> squeezed on that base (r, re)=(%.2f, %.2f), phi_d=pi/2: %.1f%%' % (b['epsbest'], 100 * b['base5'][:, 1].mean(), *b['best'][:2], 100 * b['best5'][:, 1].mean()))
print('cavity frequency at the squeezed optimum: Omega_a = %.2f  (tuned drive detuning in rev_asr2: delta = %.2f)' % (1 / np.cosh(2 * b['best'][0]), a['sel'][1]))

amplitude scan at zero squeezing (eps, fold-1 loss, fold-1 WER): [(0.1, 0.00806, 0.07), (0.2, 0.00807, 0.07), (0.3, 0.00811, 0.07), (0.4, 0.00816, 0.06), (0.5, 0.00817, 0.09), (0.7, 0.00817, 0.09), (1.0, 0.00801, 0.09)]
5-fold WER: amplitude-optimized base (eps=0.10) 12.2% -> squeezed on that base (r, re)=(0.33, 0.00), phi_d=pi/2: 10.0%
cavity frequency at the squeezed optimum: Omega_a = 0.81  (tuned drive detuning in rev_asr2: delta = -0.25)
